# KoHRM-Text-1.4B Colab T4 Smoke and Generation Test

이 노트북은 Colab T4에서 공개 `model.safetensors`를 직접 내려받아 KoHRM-Text generation이 실제로 도는지 검증합니다. `transformers.AutoModelForCausalLM`을 쓰지 않습니다. Colab의 `torchvision::nms` 충돌과 remote-code 미구현 문제를 피하기 위해 `tokenizers` + `safetensors` + 작은 PyTorch SDPA runtime을 사용합니다.

현재 공개 가중치는 사전학습 중간 체크포인트입니다. 여기서 보는 출력은 최종 assistant 품질 평가가 아니라 런타임, 토크나이저, prompt format, 반복/언어 안정성 진단입니다.

## 1. Install dependencies

In [ ]:
!pip -q install -U huggingface_hub hf_transfer safetensors "tokenizers>=0.22.0,<0.23.1"

## 2. Runtime and download settings

In [ ]:
import os
import json
import math
import subprocess
import sys
import importlib.util
from pathlib import Path

import torch
from huggingface_hub import HfApi, snapshot_download
from tokenizers import Tokenizer
from safetensors import safe_open

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_ID = "LLM-OS-Models/KoHRM-Text-1.4B"
REVISION = "main"
RUN_GENERATION = True
MAX_SEQ_LEN = 512
MAX_NEW_TOKENS = 96 if torch.cuda.is_available() else 8
TEMPERATURE = 0.0
TOP_P = 0.9
REPETITION_PENALTY = 1.18
NO_REPEAT_NGRAM_SIZE = 4

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print("gpu memory GiB:", round(free / 2**30, 2), "/", round(total / 2**30, 2))

info = HfApi().model_info(REPO_ID, revision=REVISION)
print("latest hub sha:", info.sha)

## 3. Download the latest public files

In [ ]:
patterns = [
    "README.md",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "model.safetensors",
    "kohrm_colab_generate.py",
]

repo_dir = Path(snapshot_download(
    repo_id=REPO_ID,
    revision=REVISION,
    allow_patterns=patterns,
    max_workers=8,
))

print("downloaded to:", repo_dir)
print("files:")
for path in sorted(repo_dir.iterdir()):
    if path.is_file():
        print(" -", path.name, round(path.stat().st_size / 2**20, 2), "MiB")

helper_path = repo_dir / "kohrm_colab_generate.py"
if not helper_path.exists():
    project_dir = Path("/content/KoHRM-text")
    if not project_dir.exists():
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/LLM-OS-Models/KoHRM-text",
            str(project_dir),
        ], check=True)
    helper_path = project_dir / "notebooks" / "kohrm_colab_generate.py"

print("generation helper:", helper_path)
assert helper_path.exists(), helper_path

## 4. Load config and tokenizer

KoHRM 학습 데이터는 아래 형식으로 들어갔습니다. 노트북도 같은 형식으로 prompt를 감쌉니다.

```text
<|im_start|><|object_ref_start|>instruction text<|im_end|>response text<|box_end|>
```

`<|object_ref_start|>`는 `direct` condition입니다.

In [ ]:
config = json.loads((repo_dir / "config.json").read_text())
print(json.dumps({
    "model_type": config.get("model_type"),
    "architectures": config.get("architectures"),
    "vocab_size": config.get("vocab_size"),
    "hidden_size": config.get("hidden_size"),
    "num_hidden_layers": config.get("num_hidden_layers"),
    "num_attention_heads": config.get("num_attention_heads"),
    "num_key_value_heads": config.get("num_key_value_heads"),
    "head_dim": config.get("head_dim"),
    "max_position_embeddings": config.get("max_position_embeddings"),
    "prefix_lm": config.get("prefix_lm"),
    "eos_token_id": config.get("eos_token_id"),
}, indent=2, ensure_ascii=False))

tokenizer = Tokenizer.from_file(str(repo_dir / "tokenizer.json"))
special_tokens = [
    "<|im_start|>",
    "<|im_end|>",
    "<|box_end|>",
    "<|object_ref_start|>",
    "<|object_ref_end|>",
    "<|quad_start|>",
    "<|quad_end|>",
]
special_token_ids = {tok: tokenizer.token_to_id(tok) for tok in special_tokens}

print("tokenizer vocab size:", tokenizer.get_vocab_size())
print("special token ids:")
print(json.dumps(special_token_ids, indent=2, ensure_ascii=False))

## 5. Tokenizer and prompt-format checks

In [ ]:
def format_prompt(prompt: str, condition_token: str = "<|object_ref_start|>") -> str:
    return f"<|im_start|>{condition_token}{prompt}<|im_end|>"

samples = {
    "legal_json": "다음 한국 법령/행정규칙 발췌문에서 조문명, 적용 대상, 핵심 의무를 JSON으로 추출하라.\n\n[문서명]\n진안군 홍보대사 운영 조례\n\n[조문]\n제5조 (보상)\n① 홍보대사는 무보수 명예직으로 한다.",
    "terminal_command": "한국어로 현재 디렉터리에서 용량이 큰 파일 20개를 찾고, .data 폴더는 별도로 합산하는 bash 명령을 작성하세요.",
    "finance_qa": "환율 변동이 개인 투자에 미치는 영향과 대비 전략은 무엇인가요?",
    "python_code": "def top_k_files(root, k=20):\n    return sorted(root.rglob('*'), key=lambda p: p.stat().st_size, reverse=True)[:k]",
}

rows = []
for name, text in samples.items():
    wrapped = format_prompt(text)
    ids = tokenizer.encode(wrapped, add_special_tokens=False).ids
    rows.append((name, len(text), len(ids), round(len(text) / max(1, len(ids)), 2), ids[:16]))

print(f"{'name':<18} {'chars':>8} {'tokens':>8} {'chars/token':>12} first_ids")
for name, chars, tokens, ratio, first_ids in rows:
    print(f"{name:<18} {chars:>8} {tokens:>8} {ratio:>12} {first_ids}")

## 6. Inspect `model.safetensors` before generation

In [ ]:
weights = repo_dir / "model.safetensors"
assert weights.exists(), "model.safetensors was not downloaded"

with safe_open(weights, framework="pt", device="cpu") as f:
    keys = list(f.keys())
    total_params = 0
    preview = []
    for key in keys:
        shape = tuple(f.get_slice(key).get_shape())
        n = math.prod(shape)
        total_params += n
        if len(preview) < 12:
            preview.append((key, shape, n))

print("num tensors:", len(keys))
print("num params:", f"{total_params:,}")
print("fp16/bf16 weight size estimate GiB:", round(total_params * 2 / 2**30, 2))
print("first tensors:")
for key, shape, n in preview:
    print(" -", key, shape, f"{n:,}")

## 7. Load the lightweight generation runtime once

In [ ]:
spec = importlib.util.spec_from_file_location("kohrm_colab_generate", helper_path)
kohrm = importlib.util.module_from_spec(spec)
sys.modules["kohrm_colab_generate"] = kohrm
assert spec.loader is not None
spec.loader.exec_module(kohrm)

if RUN_GENERATION:
    model, runtime_tokenizer, runtime_config = kohrm.load_kohrm(
        repo_dir,
        device="cuda" if torch.cuda.is_available() else "cpu",
        max_gpu_memory_gib=14.0,
    )
    print("loaded model device:", next(model.parameters()).device)
else:
    model = runtime_tokenizer = runtime_config = None
    print("RUN_GENERATION is False")

## 8. Run deterministic diagnostic generations

이 셀은 `temperature=0.0` greedy decoding을 기본으로 사용합니다. 반복이 심하면 `REPETITION_PENALTY`와 `NO_REPEAT_NGRAM_SIZE`를 올려 다시 실행하세요. 출력이 영어 reasoning으로 새면 런타임 문제가 아니라 현재 checkpoint가 아직 SFT/LoRA behavior alignment 전이라는 신호입니다.

In [ ]:
TEST_CASES = [
    {
        "name": "terminal_command_strict",
        "prompt": "아래 요청에 대해 bash 명령만 한 줄로 답하세요. 설명, 사고 과정, 영어 문장을 쓰지 마세요.\n\n요청: 현재 디렉터리에서 가장 큰 파일 10개를 찾는 명령",
        "max_new_tokens": 64,
    },
    {
        "name": "legal_json_strict",
        "prompt": "다음 한국 법령/행정규칙 발췌문에서 조문명, 적용 대상, 핵심 의무를 JSON 객체 하나로만 추출하세요. JSON 밖의 설명은 쓰지 마세요.\n\n[문서명]\n진안군 홍보대사 운영 조례\n\n[조문]\n제5조 (보상)\n① 홍보대사는 무보수 명예직으로 한다.\n② 군수는 홍보대사가 임무 수행을 위하여 활동하는 경우 예산의 범위 안에서 홍보활동에 직접 소요되는 실 경비로 숙식비, 차량운행 경비, 기타 비용과 격려금품을 지급할 수 있다.",
        "max_new_tokens": 128,
    },
    {
        "name": "finance_korean_short",
        "prompt": "환율 변동이 개인 투자에 미치는 영향과 대비 전략을 한국어로 4문장 이내로 설명하세요. 같은 표현을 반복하지 마세요.",
        "max_new_tokens": 96,
    },
]

if RUN_GENERATION:
    for case in TEST_CASES:
        print("\n" + "=" * 80)
        print("case:", case["name"])
        print("prompt:", case["prompt"])
        print("settings:", {
            "max_seq_len": MAX_SEQ_LEN,
            "max_new_tokens": case["max_new_tokens"],
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "repetition_penalty": REPETITION_PENALTY,
            "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
        })
        output = kohrm.generate_from_loaded(
            model,
            runtime_tokenizer,
            runtime_config,
            case["prompt"],
            max_new_tokens=case["max_new_tokens"],
            max_seq_len=MAX_SEQ_LEN,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
        )
        print("=== KoHRM output ===")
        print(output)
else:
    print("Generation skipped. Set RUN_GENERATION=True and rerun from section 7.")

## 9. Optional sampling retry

Greedy가 너무 반복적이면 아래 셀에서 낮은 temperature sampling을 비교합니다. 이 셀도 모델을 다시 로드하지 않습니다.

In [ ]:
SAMPLING_CASES = [
    "한국어로 현재 디렉터리에서 가장 큰 파일 10개를 찾는 bash 명령만 한 줄로 답하세요.",
    "환율 변동이 개인 투자에 미치는 영향과 대비 전략을 한국어로 간단히 설명하세요.",
]

if RUN_GENERATION:
    for prompt in SAMPLING_CASES:
        print("\n" + "=" * 80)
        print("prompt:", prompt)
        output = kohrm.generate_from_loaded(
            model,
            runtime_tokenizer,
            runtime_config,
            prompt,
            max_new_tokens=64,
            max_seq_len=MAX_SEQ_LEN,
            temperature=0.35,
            top_p=0.85,
            repetition_penalty=1.22,
            no_repeat_ngram_size=4,
        )
        print("=== KoHRM sampled output ===")
        print(output)
else:
    print("Generation skipped.")

## 10. Interpretation checklist

- 정상: config/tokenizer/weight inspection이 통과하고 generation cell이 OOM 없이 실행됩니다.
- 정상: 특수 토큰 ID가 `<|im_start|>=2`, `<|im_end|>=3`, `<|box_end|>=35`, `<|object_ref_start|>=32` 계열로 나옵니다.
- 주의: 영어 reasoning, 반복, JSON 필드 오류가 보이면 현재 공개 checkpoint가 아직 SFT/LoRA alignment 전이라는 뜻입니다.
- 다음 조치: `kohrm_sft_behavior_mini_v1`, `kohrm_sft_korean_domain_core_v1`, `kohrm_sft_terminal_tool_core_v1` 순서로 LoRA/SFT를 돌려 command-only, 한국어 응답, JSON fidelity를 먼저 잡습니다.